

**The Hybrid Equation: KNOWN PHYSICS + UDE**
$$ C_m \frac{dV}{dt} = I_{ext} - \mathbf{NN(V)} - I_{K}(V) - I_{L}(V) $$


In [23]:
using SciMLSensitivity
using DifferentialEquations
using SciMLSensitivity   # or DiffEqSensitivity if you prefer
using Zygote
using Optimisers         # for optimizer & update
using LinearAlgebra
using DifferentialEquations
using Flux
using Plots
using Optimization
using OptimizationOptimisers
using Zygote
using DataFrames
using  JLD2
using Random
Random.seed!(1234)
println("All the nessecessary packages have been imported")

All the nessecessary packages have been imported


In [24]:
@load "Data/synthetic_data/noise_0_hh_2d_model.jld" V # Load V (and any other relevant data)
V = Float32.(V)

501-element Vector{Float32}:
 -65.0
 -64.99959
 -64.99918
 -64.998795
 -64.99842
 -64.99806
 -64.99771
 -64.99738
 -64.99706
 -64.99676
   ⋮
 -64.78066
 -64.78989
 -64.79911
 -64.80831
 -64.81747
 -64.82657
 -64.8356
 -64.844536
 -64.85336

In [25]:
# Physics Constants (Explicit Float32)
const Cm  = 1.0f0      # µF/cm²
const g_Na = 120.0f0    # mS/cm²
const E_Na = 50.0f0     # mV
const g_L  = 0.3f0      # mS/cm²
const E_L  = -54.387f0  # mV
const g_K  = 36.0f0    #potassium conductance (mS/cm²)
const E_K  = -77.0f0 

-77.0f0

In [26]:
# Voltage-gated ion channel kinetics (Float32)

α_n(V::Float32) = 0.01f0 * (V + 55f0) / (1f0 - exp(-(V + 55f0) / 10f0))
β_n(V::Float32) = 0.125f0 * exp(-(V + 65f0) / 80f0)

α_m(V::Float32) = 0.1f0 * (V + 40f0) / (1f0 - exp(-(V + 40f0) / 10f0))
β_m(V::Float32) = 4.0f0 * exp(-(V + 65f0) / 18f0)

α_h(V::Float32) = 0.07f0 * exp(-(V + 65f0) / 20f0)
β_h(V::Float32) = 1f0 / (1f0 + exp(-(V + 35f0) / 10f0))

# Steady-state & time-constant functions (Float32)

m_inf(V::Float32) = α_m(V) / (α_m(V) + β_m(V))
h_inf(V::Float32) = α_h(V) / (α_h(V) + β_h(V))
n_inf(V::Float32) = α_n(V) / (α_n(V) + β_n(V))

tau_n(V::Float32) = 1f0 / (α_n(V) + β_n(V))

println("Physics of neural dynamics defined in Float32")


Physics of neural dynamics defined in Float32


In [27]:
NN_Model = Chain(
    Dense(1, 16 , tanh), # Inputs: V and n,
    Dense(16, 1)
)|> Flux.f32

Chain(
  Dense(1 => 16, tanh),                 # 32 parameters
  Dense(16 => 1),                       # 17 parameters
)                   # Total: 4 arrays, 49 parameters, 404 bytes.

In [28]:
p_nn, re = Flux.destructure(NN_Model)
p_nn = Float32.(p_nn)
println("Recruit Constructed. Parameters: ", length(p_nn))
println("Parameter element type: ", eltype(p_nn))


Recruit Constructed. Parameters: 49
Parameter element type: Float32


In [29]:
function Stimulus(t::Float32)
    return (t >= 10f0 && t < 11f0) ? 20f0 : 0f0
end

Stimulus (generic function with 1 method)

In [30]:
using  Statistics
const V_mean  = mean(V)
const V_std = std(V)
norm_input(v)=(v-V_mean)/V_std



norm_input (generic function with 1 method)

In [31]:
function hodgkin_huxley_UDE!(du,u,p,t)
          V,n = u 



          nn_model = re(p)
          pred_I_Na = nn_model([norm_input(V)])[1]
          I_ext = Stimulus(t)
          I_K   = g_K * n^4 * (V - E_K)
          I_L   = g_L * (V - E_L)
          # Dynamics
    du[1] = (I_ext - (pred_I_Na + I_K + I_L) / Cm)
    du[2] = (n_inf(V) - n) / tau_n(V)
end


u0_true = [-65.0f0, n_inf(-65.0f0)]

t_span = (0.0f0, 50.0f0)
const t_train = range(t_span[1], t_span[2], length=length(V))  
prob_nn = ODEProblem(hodgkin_huxley_UDE! , u0_true,t_span , p_nn)

ODEProblem with uType Vector{Float32} and tType Float32. In-place: true
Non-trivial mass matrix: false
timespan: (0.0f0, 50.0f0)
u0: 2-element Vector{Float32}:
 -65.0
   0.3176769

In [32]:
t_train

0.0f0:0.1f0:50.0f0

In [33]:
function predict_ude(p)
    p_safe = Float32.(p)
    _prob = remake(prob_nn, p=p_safe)
    solve(_prob,
          Rodas5P(),   # use finite-diff Jacobian
          saveat = Float32.(collect(t_train)),
          reltol = 1f-6, abstol = 1f-6,
          sensealg = InterpolatingAdjoint(autojacvec = ZygoteVJP()))
end

predict_ude (generic function with 1 method)

In [34]:
function loss(p)
    pred = predict_ude(p)
    if pred.retcode != :Success
        return 1e6
    end
    pred_V = pred[1, :]
    phys_loss = sum(abs2, pred_V - V)
    return phys_loss
    
end

loss (generic function with 1 method)

In [35]:
lossess  = Float32[]
min_loss = Inf
const  SAVE_PATH = "Data/Loss_and_Parameters/ude_best_model_noise_0.jld2"

"Data/Loss_and_Parameters/ude_best_model_noise_0.jld2"

In [36]:
using Printf
losses = Float64[]         
min_loss = Inf              
const SAVE_PATH = "ude_best_model.jld2"

# --------------------------------------------------
# ROBUST CALLBACK FOR SciML TRAINING
# --------------------------------------------------
function callback(state, l)
    # 1. Store loss
    push!(losses, l)
    iter = length(losses)

    # 2. Safety check: stop if loss explodes
    if isnan(l)
        println("🔥 LOSS EXPLODED (NaN). STOPPING TRAINING.")
        return true
    end

    # 3. Check if this is the best model so far
    is_new_best = false
    if l < min_loss
        global min_loss = l
        is_new_best = true

        # 4. Save best model only (checkpoint)
        jldsave(
            SAVE_PATH;
            params = state.u,          # NN parameters
            loss_history = losses,     # full loss curve
            best_loss = l
        )
    end

    # 5. Print progress
    if iter % 10 == 0 || is_new_best
        @printf("Iter: %4d | Loss: %.5e", iter, l)
        if is_new_best
            print(" 🌟 NEW BEST! (Saved)")
        end
        println()
    end

    # 6. Continue training
    return false
end


callback (generic function with 1 method)

In [37]:
# define optf after loss
optf = Optimization.OptimizationFunction((x, p) -> loss(x), Optimization.AutoZygote())
optprob = Optimization.OptimizationProblem(optf, p_nn)
println("Optimization Function (optf) Defined (Protocol Zero).")

Optimization Function (optf) Defined (Protocol Zero).


In [38]:
res_adam = Optimization.solve(
    optprob,
    OptimizationOptimisers.Adam(0.05),
    callback = callback,
    maxiters = 1000
)

OrdinaryDiffEqDifferentiation.FirstAutodiffJacError: First call to automatic differentiation for the Jacobian
failed. This means that the user `f` function is not compatible
with automatic differentiation. Methods to fix this include:

1. Turn off automatic differentiation (e.g. Rosenbrock23() becomes
   Rosenbrock23(autodiff = AutoFiniteDiff())). More details can befound at
   https://docs.sciml.ai/DiffEqDocs/stable/features/performance_overloads/
2. Improving the compatibility of `f` with ForwardDiff.jl automatic
   differentiation (using tools like PreallocationTools.jl). More details
   can be found at https://docs.sciml.ai/DiffEqDocs/stable/basics/faq/#Autodifferentiation-and-Dual-Numbers
3. Defining analytical Jacobians. More details can be
   found at https://docs.sciml.ai/DiffEqDocs/stable/types/ode_types/#SciMLBase.ODEFunction

Note: turning off automatic differentiation tends to have a very minimal
performance impact (for this use case, because it's forward mode for a
square 